In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
real = pd.read_csv("True.csv")
fake = pd.read_csv("Fake.csv")

real["label"] = 1
fake["label"] = 0

data = pd.concat([real, fake], ignore_index=True)

data["text"] = data["text"].fillna("")
data["subject"] = data["subject"].fillna("Unknown")

data.head()

,title,text,subject,date,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    data["text"],
    data["label"],
    test_size=0.2,
    random_state=42,
    stratify=data["label"]
)

train_subject = data.loc[X_train.index, "subject"]
test_subject = data.loc[X_test.index, "subject"]

In [4]:
model_before = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000)),
    ("lr", LogisticRegression())
])

model_before.fit(X_train, y_train)

pred_before = model_before.predict(X_test)

In [5]:
# Check Bias Across News Subjects
before_data = pd.DataFrame({
    "subject": test_subject,
    "actual": y_test,
    "predicted": pred_before
})

before_results = before_data.groupby("subject").apply(
    lambda x: pd.Series({
        "Articles": len(x),
        "Accuracy": accuracy_score(x["actual"], x["predicted"]),
        "Fake Prediction %": (x["predicted"] == 0).mean() * 100
    }),
    include_groups=False
)

before_results

,Articles,Accuracy,Fake Prediction %
subject,,,
Government News,318.0,0.949686,94.968553
Middle-east,173.0,0.988439,98.843931
News,1771.0,0.998871,99.887069
US_News,176.0,0.982955,98.295455
left-news,904.0,0.978982,97.898230
politics,1354.0,0.979321,97.932053
politicsNews,2196.0,0.984973,1.502732
worldnews,2088.0,0.993774,0.622605


In [6]:
# Apply Bias Mitigation by Balancing Subjects
train_data = data.loc[X_train.index].copy()

subject_count = train_data["subject"].value_counts().min()

balanced_train = train_data.groupby("subject").sample(
    n=subject_count, random_state=42
)

X_balanced = balanced_train["text"]
y_balanced = balanced_train["label"]

In [7]:
# Train Model After Mitigation
model_after = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000)),
    ("lr", LogisticRegression())
])

model_after.fit(X_balanced, y_balanced)

pred_after = model_after.predict(X_test)

In [8]:
# Check Bias After Mitigation
after_data = pd.DataFrame({
    "subject": test_subject,
    "actual": y_test,
    "predicted": pred_after
})

after_results = after_data.groupby("subject").apply(
    lambda x: pd.Series({
        "Articles": len(x),
        "Accuracy": accuracy_score(x["actual"], x["predicted"]),
        "Fake Prediction %": (x["predicted"] == 0).mean() * 100
    })
)

after_results

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_18912\2523852133.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  after_results = after_data.groupby("subject").apply(


,Articles,Accuracy,Fake Prediction %
subject,,,
Government News,318.0,0.974843,97.484277
Middle-east,173.0,0.994220,99.421965
News,1771.0,0.999435,99.943535
US_News,176.0,1.000000,100.000000
left-news,904.0,0.991150,99.115044
politics,1354.0,0.992614,99.261448
politicsNews,2196.0,0.891621,10.837887
worldnews,2088.0,0.964080,3.591954


In [9]:
# Compare Before and After Mitigation
comparison = pd.concat(
    [before_results.add_prefix("Before_"),
     after_results.add_prefix("After_")],
    axis=1
)

comparison

,Before_Articles,Before_Accuracy,Before_Fake Prediction %,After_Articles,After_Accuracy,After_Fake Prediction %
subject,,,,,,
Government News,318.0,0.949686,94.968553,318.0,0.974843,97.484277
Middle-east,173.0,0.988439,98.843931,173.0,0.994220,99.421965
News,1771.0,0.998871,99.887069,1771.0,0.999435,99.943535
US_News,176.0,0.982955,98.295455,176.0,1.000000,100.000000
left-news,904.0,0.978982,97.898230,904.0,0.991150,99.115044
politics,1354.0,0.979321,97.932053,1354.0,0.992614,99.261448
politicsNews,2196.0,0.984973,1.502732,2196.0,0.891621,10.837887
worldnews,2088.0,0.993774,0.622605,2088.0,0.964080,3.591954
